In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [4]:
file_path = 'HIT140_World_Cup_2026_FINAL.xlsx'

# Load Objective 2.1
df_2_1 = pd.read_excel(file_path, sheet_name='Objective_2.1')
predictors_2_1 = ['Elo_Ranking_Difference', 'FIFA_Ranking_Difference', 
                   'Avg_Goals_Scored_Difference', 'Avg_Goals_Conceded_Difference',
                   'Squad_Elo_Interaction', 'Goals_Form_Interaction', 
                   'Head_to_Head_Record', 'Tournament_Stage']
X_2_1 = df_2_1[predictors_2_1].values
y_2_1 = df_2_1['Goal_Difference'].values

# Load Objective 2.2
df_2_2 = pd.read_excel(file_path, sheet_name='Objective_2.2')
predictors_2_2 = ['Elo_Rating', 'FIFA_Ranking', 'Avg_Goals_Scored', 
                   'Avg_Goals_Conceded', 'Squad_Value', 'Opponent_FIFA_Ranking',
                   'Opponent_Avg_Goals_Conceded', 'Team_Rest_Days']
X_2_2 = df_2_2[predictors_2_2].values
y_2_2 = df_2_2['Goals_Scored'].values

print(f"Objective 2.1: {X_2_1.shape[0]} matches, {X_2_1.shape[1]} predictors")
print(f"Objective 2.2: {X_2_2.shape[0]} team-matches, {X_2_2.shape[1]} predictors")

Objective 2.1: 104 matches, 8 predictors
Objective 2.2: 208 team-matches, 8 predictors


In [3]:
%pip install openpyxl -q

In [5]:
# Objective 2.1
X_train_2_1, X_test_2_1, y_train_2_1, y_test_2_1 = train_test_split(
    X_2_1, y_2_1, test_size=0.2, random_state=42
)

# Objective 2.2
X_train_2_2, X_test_2_2, y_train_2_2, y_test_2_2 = train_test_split(
    X_2_2, y_2_2, test_size=0.2, random_state=42
)

print(f"Objective 2.1: {len(X_train_2_1)} train, {len(X_test_2_1)} test")
print(f"Objective 2.2: {len(X_train_2_2)} train, {len(X_test_2_2)} test")

Objective 2.1: 83 train, 21 test
Objective 2.2: 166 train, 42 test


In [6]:
# Objective 2.1
scaler_2_1 = StandardScaler()
X_train_2_1_scaled = scaler_2_1.fit_transform(X_train_2_1)
X_test_2_1_scaled = scaler_2_1.transform(X_test_2_1)

# Objective 2.2
scaler_2_2 = StandardScaler()
X_train_2_2_scaled = scaler_2_2.fit_transform(X_train_2_2)
X_test_2_2_scaled = scaler_2_2.transform(X_test_2_2)

print("Features standardized")

Features standardized


In [7]:
lr_2_1 = LinearRegression()
lr_2_1.fit(X_train_2_1_scaled, y_train_2_1)

# Predictions
train_pred_lr_2_1 = lr_2_1.predict(X_train_2_1_scaled)
test_pred_lr_2_1 = lr_2_1.predict(X_test_2_1_scaled)

# Metrics
train_r2_lr_2_1 = r2_score(y_train_2_1, train_pred_lr_2_1)
test_r2_lr_2_1 = r2_score(y_test_2_1, test_pred_lr_2_1)
test_mae_lr_2_1 = mean_absolute_error(y_test_2_1, test_pred_lr_2_1)

print("Objective 2.1 - Linear Regression:")
print(f"  Train R²: {train_r2_lr_2_1:.4f}")
print(f"  Test R²:  {test_r2_lr_2_1:.4f}")
print(f"  Test MAE: {test_mae_lr_2_1:.4f}")
print(f"  Gap:      {train_r2_lr_2_1 - test_r2_lr_2_1:.4f}")

Objective 2.1 - Linear Regression:
  Train R²: 0.3371
  Test R²:  0.1209
  Test MAE: 1.2181
  Gap:      0.2162


In [8]:
poly_2_1 = PolynomialFeatures(degree=2, include_bias=False)
X_train_2_1_poly = poly_2_1.fit_transform(X_train_2_1_scaled)
X_test_2_1_poly = poly_2_1.transform(X_test_2_1_scaled)

lr_poly_2_1 = LinearRegression()
lr_poly_2_1.fit(X_train_2_1_poly, y_train_2_1)

# Predictions
train_pred_poly_2_1 = lr_poly_2_1.predict(X_train_2_1_poly)
test_pred_poly_2_1 = lr_poly_2_1.predict(X_test_2_1_poly)

# Metrics
train_r2_poly_2_1 = r2_score(y_train_2_1, train_pred_poly_2_1)
test_r2_poly_2_1 = r2_score(y_test_2_1, test_pred_poly_2_1)
test_mae_poly_2_1 = mean_absolute_error(y_test_2_1, test_pred_poly_2_1)

print("Objective 2.1 - Polynomial Features (8 → 44 features):")
print(f"  Train R²: {train_r2_poly_2_1:.4f}")
print(f"  Test R²:  {test_r2_poly_2_1:.4f}")
print(f"  Test MAE: {test_mae_poly_2_1:.4f}")
print(f"  Gap:      {train_r2_poly_2_1 - test_r2_poly_2_1:.4f}")
print(f"  ⚠️  Test R² is NEGATIVE - Model overfits!")

Objective 2.1 - Polynomial Features (8 → 44 features):
  Train R²: 0.6926
  Test R²:  -2.1841
  Test MAE: 2.0673
  Gap:      2.8767
  ⚠️  Test R² is NEGATIVE - Model overfits!


In [9]:
ridge_2_1 = Ridge(alpha=1.0)
ridge_2_1.fit(X_train_2_1_scaled, y_train_2_1)

# Predictions
train_pred_ridge_2_1 = ridge_2_1.predict(X_train_2_1_scaled)
test_pred_ridge_2_1 = ridge_2_1.predict(X_test_2_1_scaled)

# Metrics
train_r2_ridge_2_1 = r2_score(y_train_2_1, train_pred_ridge_2_1)
test_r2_ridge_2_1 = r2_score(y_test_2_1, test_pred_ridge_2_1)
test_mae_ridge_2_1 = mean_absolute_error(y_test_2_1, test_pred_ridge_2_1)

print("Objective 2.1 - Ridge Regression:")
print(f"  Train R²: {train_r2_ridge_2_1:.4f}")
print(f"  Test R²:  {test_r2_ridge_2_1:.4f}")
print(f"  Test MAE: {test_mae_ridge_2_1:.4f}")
print(f"  Gap:      {train_r2_ridge_2_1 - test_r2_ridge_2_1:.4f}")

Objective 2.1 - Ridge Regression:
  Train R²: 0.3367
  Test R²:  0.1265
  Test MAE: 1.2154
  Gap:      0.2103


In [10]:
lr_2_2 = LinearRegression()
lr_2_2.fit(X_train_2_2_scaled, y_train_2_2)

train_pred_lr_2_2 = lr_2_2.predict(X_train_2_2_scaled)
test_pred_lr_2_2 = lr_2_2.predict(X_test_2_2_scaled)

train_r2_lr_2_2 = r2_score(y_train_2_2, train_pred_lr_2_2)
test_r2_lr_2_2 = r2_score(y_test_2_2, test_pred_lr_2_2)
test_mae_lr_2_2 = mean_absolute_error(y_test_2_2, test_pred_lr_2_2)

print("\nObjective 2.2 - Linear Regression:")
print(f"  Train R²: {train_r2_lr_2_2:.4f}")
print(f"  Test R²:  {test_r2_lr_2_2:.4f}")
print(f"  Test MAE: {test_mae_lr_2_2:.4f}")
print(f"  Gap:      {train_r2_lr_2_2 - test_r2_lr_2_2:.4f}")


Objective 2.2 - Linear Regression:
  Train R²: 0.3148
  Test R²:  0.3759
  Test MAE: 0.8177
  Gap:      -0.0611


In [11]:
poly_2_2 = PolynomialFeatures(degree=2, include_bias=False)
X_train_2_2_poly = poly_2_2.fit_transform(X_train_2_2_scaled)
X_test_2_2_poly = poly_2_2.transform(X_test_2_2_scaled)

lr_poly_2_2 = LinearRegression()
lr_poly_2_2.fit(X_train_2_2_poly, y_train_2_2)

train_pred_poly_2_2 = lr_poly_2_2.predict(X_train_2_2_poly)
test_pred_poly_2_2 = lr_poly_2_2.predict(X_test_2_2_poly)

train_r2_poly_2_2 = r2_score(y_train_2_2, train_pred_poly_2_2)
test_r2_poly_2_2 = r2_score(y_test_2_2, test_pred_poly_2_2)
test_mae_poly_2_2 = mean_absolute_error(y_test_2_2, test_pred_poly_2_2)

print("Objective 2.2 - Polynomial Features:")
print(f"  Train R²: {train_r2_poly_2_2:.4f}")
print(f"  Test R²:  {test_r2_poly_2_2:.4f}")
print(f"  Test MAE: {test_mae_poly_2_2:.4f}")
print(f"  Gap:      {train_r2_poly_2_2 - test_r2_poly_2_2:.4f}")
print(f"  ⚠️  Test R² is NEGATIVE - Model overfits!")

Objective 2.2 - Polynomial Features:
  Train R²: 0.5266
  Test R²:  -0.0855
  Test MAE: 1.0834
  Gap:      0.6121
  ⚠️  Test R² is NEGATIVE - Model overfits!


In [12]:
ridge_2_2 = Ridge(alpha=1.0)
ridge_2_2.fit(X_train_2_2_scaled, y_train_2_2)

train_pred_ridge_2_2 = ridge_2_2.predict(X_train_2_2_scaled)
test_pred_ridge_2_2 = ridge_2_2.predict(X_test_2_2_scaled)

train_r2_ridge_2_2 = r2_score(y_train_2_2, train_pred_ridge_2_2)
test_r2_ridge_2_2 = r2_score(y_test_2_2, test_pred_ridge_2_2)
test_mae_ridge_2_2 = mean_absolute_error(y_test_2_2, test_pred_ridge_2_2)

print("Objective 2.2 - Ridge Regression:")
print(f"  Train R²: {train_r2_ridge_2_2:.4f}")
print(f"  Test R²:  {test_r2_ridge_2_2:.4f}")
print(f"  Test MAE: {test_mae_ridge_2_2:.4f}")
print(f"  Gap:      {train_r2_ridge_2_2 - test_r2_ridge_2_2:.4f}")

Objective 2.2 - Ridge Regression:
  Train R²: 0.3148
  Test R²:  0.3752
  Test MAE: 0.8173
  Gap:      -0.0604


In [13]:
print("\n" + "="*70)
print("FINAL RESULTS - TEST SET PERFORMANCE (Most Important)")
print("="*70)

print("\nObjective 2.1 (Goal Difference):")
print(f"  Linear:     Test R² = {test_r2_lr_2_1:.4f}     ✓ WINNER")
print(f"  Polynomial: Test R² = {test_r2_poly_2_1:.4f}   ✗ Overfits")
print(f"  Ridge:      Test R² = {test_r2_ridge_2_1:.4f}")

print("\nObjective 2.2 (Goals Scored):")
print(f"  Linear:     Test R² = {test_r2_lr_2_2:.4f}     ✓ WINNER")
print(f"  Polynomial: Test R² = {test_r2_poly_2_2:.4f}   ✗ Overfits")
print(f"  Ridge:      Test R² = {test_r2_ridge_2_2:.4f}")

print("\n" + "="*70)
print("RECOMMENDATION: USE LINEAR REGRESSION FOR BOTH OBJECTIVES")
print("="*70)
print("""
✓ Linear regression has best test R² (0.1209 and 0.3759)
✓ Stable generalization gaps (0.22 and -0.06)
✗ Polynomial features overfit (negative test R²)
✓ Ridge matches linear (confirms feature dimensionality issue)
""")


FINAL RESULTS - TEST SET PERFORMANCE (Most Important)

Objective 2.1 (Goal Difference):
  Linear:     Test R² = 0.1209     ✓ WINNER
  Polynomial: Test R² = -2.1841   ✗ Overfits
  Ridge:      Test R² = 0.1265

Objective 2.2 (Goals Scored):
  Linear:     Test R² = 0.3759     ✓ WINNER
  Polynomial: Test R² = -0.0855   ✗ Overfits
  Ridge:      Test R² = 0.3752

RECOMMENDATION: USE LINEAR REGRESSION FOR BOTH OBJECTIVES

✓ Linear regression has best test R² (0.1209 and 0.3759)
✓ Stable generalization gaps (0.22 and -0.06)
✗ Polynomial features overfit (negative test R²)
✓ Ridge matches linear (confirms feature dimensionality issue)

